In [1]:
import os
# This single line fixes BOTH the dense and sparse attention modules!
os.environ['ATTN_BACKEND'] = 'xformers' 
os.environ['SPCONV_ALGO'] = 'native'

import torch
import torch.nn.functional as F

class TrellisKVCacheManager:
    def __init__(self, model):
        self.model = model
        self.kv_cache = {}
        self.use_cache = False
        self.use_spatial_blend = False
        self.external_grid_mask = None  # NEW: Holds the mask passed from the outside
        self.call_counters = {}
        self.hooks = []

    def register_hooks(self):
        for name, module in self.model.named_modules():
            if name.endswith('self_attn.to_qkv'):
                hook = module.register_forward_hook(
                    lambda m, inp, out, n=name: self._self_attn_hook(n, out)
                )
                self.hooks.append(hook)
            # elif name.endswith('cross_attn.to_kv'):
            #     hook = module.register_forward_hook(
            #         lambda m, inp, out, n=name: self._cross_attn_hook(n, out)
            #     )
            #     self.hooks.append(hook)

    def set_spatial_mask(self, bbox_dict):
        """
        Accepts a dictionary: {'min': [z, y, x], 'max': [z, y, x]}
        Values are normalized between 0.0 and 1.0.
        """
        self.external_grid_mask = bbox_dict

    def _resolve_spatial_mask(self, tensor, is_sparse, output_obj):
        if self.external_grid_mask is None:
            return torch.ones_like(tensor[..., :1])
            
        bbox = self.external_grid_mask
        z_min, y_min, x_min = bbox['min']
        z_max, y_max, x_max = bbox['max']

        if is_sparse:
            # 1. Get raw float coordinates [N, 3] -> (z, y, x)
            coords = output_obj.indices.float()[:, 1:4] 
            spatial_shape = torch.tensor(output_obj.spatial_shape, device=coords.device, dtype=coords.dtype)
            
            # 2. Normalize coordinates into a clean [0, 1] range
            norm_coords = coords / (spatial_shape - 1.0)
            
            # 3. Check which points fall INSIDE the bounding box limits
            mask_cond = (
                (norm_coords[:, 0] >= z_min) & (norm_coords[:, 0] <= z_max) &
                (norm_coords[:, 1] >= y_min) & (norm_coords[:, 1] <= y_max) &
                (norm_coords[:, 2] >= x_min) & (norm_coords[:, 2] <= x_max)
            )
            return mask_cond.to(tensor.dtype).unsqueeze(-1) # Shape: [N, 1]
            
        else:
            S = tensor.shape[-2] 
            G = int(round(S**(1/3))) 
            
            # 1. Recreate the 3D voxel grid indices for the sequence
            idx = torch.arange(S, device=tensor.device)
            z_idx = idx % G
            y_idx = (idx // G) % G
            x_idx = idx // (G * G)
            
            # Safe protection against single-voxel layer sizes
            denom = float(G - 1) if G > 1 else 1.0

            # 2. Normalize grid positions into a [0, 1] range
            y_norm = y_idx.float() / denom
            z_norm = z_idx.float() / denom
            x_norm = x_idx.float() / denom
            
            # 3. Apply the bounding box criteria to the flattened grid
            mask_cond = (
                (z_norm >= z_min) & (z_norm <= z_max) &
                (y_norm >= y_min) & (y_norm <= y_max) &
                (x_norm >= x_min) & (x_norm <= x_max)
            )
            
            W = mask_cond.to(tensor.dtype).unsqueeze(-1) # Shape: [S, 1]
            return W.expand(tensor.shape[:-1] + (1,))

    
    def _align_and_blend(self, new_tensor, cache_tensor, W):
        # Because we force coordinates to match, min_len truncation is no longer 
        # a destructive hack. The sequences will match perfectly.
        min_len = min(new_tensor.shape[-2], cache_tensor.shape[-2])
        t_new = new_tensor[..., :min_len, :]
        t_cache = cache_tensor[..., :min_len, :]
        w_align = W[..., :min_len, :]
            
        blended = w_align * t_new + (1 - w_align) * t_cache
        
        out = new_tensor.clone()
        out[..., :min_len, :] = blended
        return out

    def _self_attn_hook(self, name, output):
        step = self.call_counters.get(name, 0)
        self.call_counters[name] = step + 1
        cache_key = f"{name}_step_{step}"
        is_sparse = hasattr(output, 'feats')
        tensor = output.feats if is_sparse else output

        if not self.use_cache:
            q, k, v = tensor.chunk(3, dim=-1)
            self.kv_cache[cache_key] = (k.detach().to(torch.float16).cpu(), 
                                        v.detach().to(torch.float16).cpu())
            return output
        else:
            cached_item = self.kv_cache.get(cache_key)
            if not cached_item or not isinstance(cached_item, tuple):
                return output

            k_cached, v_cached = cached_item
            k_cached = k_cached.to(tensor.device).to(tensor.dtype)
            v_cached = v_cached.to(tensor.device).to(tensor.dtype)
            q, k_new, v_new = tensor.chunk(3, dim=-1)

            if self.use_spatial_blend:
                W = self._resolve_spatial_mask(tensor, is_sparse, output)
                k_final = self._align_and_blend(k_new, k_cached, W)
                v_final = self._align_and_blend(v_new, v_cached, W)
            else:
                k_final, v_final = k_cached, v_cached

            new_tensor = torch.cat([q, k_final, v_final], dim=-1)
            return output.replace(new_tensor) if is_sparse else new_tensor

    def _cross_attn_hook(self, name, output):
        step = self.call_counters.get(name, 0)
        self.call_counters[name] = step + 1
        cache_key = f"{name}_step_{step}"
        is_sparse = hasattr(output, 'feats')
        tensor = output.feats if is_sparse else output

        if not self.use_cache:
            self.kv_cache[cache_key] = tensor.detach().to(torch.float16).cpu()
            return output
        else:
            # 🚨 THE FIX: If we are blending, DO NOT touch cross attention!
            # Let the flow network see image_2's conditioning natively.
            if self.use_spatial_blend:
                return output
                
            # (Only do full replace if we are doing a strict Sanity Check)
            kv_cached = self.kv_cache.get(cache_key)
            if kv_cached is None or isinstance(kv_cached, tuple):
                return output
            
            kv_cached = kv_cached.to(tensor.device).to(tensor.dtype)
            return output.replace(kv_cached) if is_sparse else kv_cached

    def reset_counters(self):
        self.call_counters = {}

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []

In [2]:
from TRELLIS.trellis.pipelines import TrellisImageTo3DPipeline
from TRELLIS.trellis.utils import postprocessing_utils, render_utils
from config import TRAINING_CONFIG
from PIL import Image
import gc

image_1 = Image.open(
    "editing/samples/gen_img11.png"
).convert("RGB")
image_2 = Image.open(
    "editing/samples/gen_img22.png"
).convert("RGB")

device = TRAINING_CONFIG["device"]
pipeline = TrellisImageTo3DPipeline.from_pretrained(TRAINING_CONFIG["model_backbone"])
pipeline.to(device)

# Create the manager for both generation stages with correct TRELLIS keys
ss_manager = TrellisKVCacheManager(pipeline.models['sparse_structure_flow_model'])
slat_manager = TrellisKVCacheManager(pipeline.models['slat_flow_model'])

# Attach the hooks
ss_manager.register_hooks()
slat_manager.register_hooks()

# ==========================================
# INFERENCE 1: Generate & Cache
# ==========================================
ss_manager.reset_counters()
slat_manager.reset_counters()
ss_manager.use_cache = False
slat_manager.use_cache = False

# Run the pipeline with the first image and a fixed seed
outputs_1 = pipeline.run(image_1, seed=123)

# Export the first mesh
glb_1 = postprocessing_utils.to_glb(
    outputs_1['gaussian'][0],
    outputs_1['mesh'][0],
    simplify=0.95,
    texture_size=1024,
)
glb_1.export("kv_cache/mesh_pass1.glb")

torch.cuda.empty_cache()
gc.collect()

# ==========================================
# INFERENCE 2: Replace Entirely
# ==========================================
ss_manager.reset_counters()
slat_manager.reset_counters()
ss_manager.use_cache = True
slat_manager.use_cache = True
ss_manager.use_spatial_blend = True
slat_manager.use_spatial_blend = True

# MY_BBOX_MASK = {
#     'min': [0.0, 0.0, 0.6],
#     'max': [0.45, 0.4, 1.0]
# }
MY_BBOX_MASK = {
    'min': [0.0, 0.2, 0.0],
    'max': [1.0, 0.8, 1.0]
}

ss_manager.set_spatial_mask(MY_BBOX_MASK)
slat_manager.set_spatial_mask(MY_BBOX_MASK)

# Run the pipeline with the SECOND image, but the EXACT SAME seed
outputs_2 = pipeline.run(image_2, seed=123)

# Export the second mesh
glb_2 = postprocessing_utils.to_glb(
    outputs_2['gaussian'][0],
    outputs_2['mesh'][0],
    simplify=0.95,
    texture_size=1024,
)
glb_2.export("kv_cache/mesh_pass2.glb")

# ==========================================
# VERIFICATION
# ==========================================
# Compare the raw PyTorch tensors to guarantee they are 100% identical
vertices_match = torch.equal(outputs_1['mesh'][0].vertices, outputs_2['mesh'][0].vertices)
faces_match = torch.equal(outputs_1['mesh'][0].faces, outputs_2['mesh'][0].faces)

# print(f"Are vertices mathematically identical? {vertices_match}")
# print(f"Are faces mathematically identical? {faces_match}")

[SPARSE] Backend: spconv, Attention: xformers
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[SPARSE][CONV] spconv algo: native
[ATTENTION] Using backend: xformers


c:\Users\nov1\AppData\Local\miniconda3\envs\trellis\Lib\site-packages\spconv\pytorch\functional.py:51: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  _TORCH_CUSTOM_FWD = amp.custom_fwd(cast_inputs=torch.float16)
c:\Users\nov1\AppData\Local\miniconda3\envs\trellis\Lib\site-packages\spconv\pytorch\functional.py:100: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @_TORCH_CUSTOM_BWD
c:\Users\nov1\AppData\Local\miniconda3\envs\trellis\Lib\site-packages\spconv\pytorch\functional.py:166: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @_TORCH_CUSTOM_BWD
c:\Users\nov1\AppData\Local\miniconda3\envs\trellis\Lib\site-packages\spconv\pytorch\functional.py:246: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is depreca

Before postprocess: 176302 vertices, 352656 faces


Decimating Mesh: 100%|██████████[00:03<00:00]


After decimate: 8791 vertices, 17632 faces


...

Decimating Mesh: 100%|██████████[00:03<00:00]


After decimate: 9008 vertices, 18027 faces


Rasterizing: 100%|██████████| 1000/1000 [00:01<00:00, 570.36it/s]


Found 5662 invisible faces
Dual graph: 27034 edges
Mincut solved, start checking the cut
Removed 7328 faces by mincut
After remove invisible faces: 5472 vertices, 10968 faces


Rendering: 100it [00:01, 50.13it/s]
Texture baking (opt): optimizing: 100%|██████████| 2500/2500 [00:14<00:00, 174.79it/s, loss=0.0114]
